# 📊 Telco Customer Churn Analysis
**Objective:** Analyze customer behavior, identify key churn drivers, build a prediction model, and provide business recommendations.

---
**Dataset:** Telco Customer Churn (7,043 customers, 33 features)

**Workflow:**
1. Data Loading & Overview
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Model Building
6. Business Recommendations

## 1️⃣ Data Loading & Overview

In [ ]:
# ── Core Libraries ──────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization ────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Machine Learning ─────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_auc_score, roc_curve
)

# ── Settings ─────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 50)

print('✅ All libraries loaded successfully!')

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────
# UPDATE this path to where your CSV file is saved
df = pd.read_csv('Telco_customer_churn.csv')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Basic Info ───────────────────────────────────────────────
df.info()

In [ ]:
# ── Statistical Summary ──────────────────────────────────────
df.describe(include='all').T

## 2️⃣ Data Cleaning

In [ ]:
# ── Check Missing Values ─────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

In [ ]:
# ── Drop Irrelevant / Geo Columns ────────────────────────────
cols_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City',
                'Zip Code', 'Lat Long', 'Latitude', 'Longitude',
                'Churn Score', 'CLTV', 'Churn Reason']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

# ── Fix Target Column ────────────────────────────────────────
# Keep 'Churn Value' (0/1) as target; drop text label
df.drop(columns=['Churn Label'], inplace=True, errors='ignore')
df.rename(columns={'Churn Value': 'Churn'}, inplace=True)

# ── Fix Total Charges (sometimes stored as string) ───────────
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'].fillna(df['Total Charges'].median(), inplace=True)

# ── Strip Whitespace from String Columns ─────────────────────
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

print(f'✅ Cleaned shape: {df.shape}')
df.head(3)

In [ ]:
# ── Check Duplicates ─────────────────────────────────────────
print(f'Duplicate rows: {df.duplicated().sum()}')

## 3️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Overall Churn Distribution ───────────────────────────
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
axes[0].bar(['No Churn', 'Churned'], churn_counts,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Count', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts, labels=['No Churn', 'Churned'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0, 0.05))
axes[1].set_title('Churn Rate', fontsize=14, fontweight='bold')

plt.suptitle('Customer Churn Overview', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n📌 Churn Rate: {churn_pct[1]:.1f}%  |  Retention Rate: {churn_pct[0]:.1f}%')

In [ ]:
# ── 3.2 Churn by Contract Type ───────────────────────────────
contract_churn = df.groupby('Contract')['Churn'].mean() * 100

ax = contract_churn.sort_values(ascending=False).plot(
    kind='bar', color=['#e74c3c', '#f39c12', '#2ecc71'],
    edgecolor='white', figsize=(9, 5)
)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate (%)')
plt.xticks(rotation=0)

for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Tenure Distribution by Churn ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE Plot
for label, grp in df.groupby('Churn'):
    grp['Tenure Months'].plot(kind='kde', ax=axes[0],
                               label='Churned' if label else 'Not Churned')
axes[0].set_title('Tenure Distribution (KDE)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tenure (Months)')
axes[0].legend()

# Box Plot
df.boxplot(column='Tenure Months', by='Churn', ax=axes[1],
           patch_artist=True,
           boxprops=dict(facecolor='#3498db', color='#2c3e50'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Tenure by Churn (Boxplot)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Churn (0=No, 1=Yes)')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Monthly & Total Charges vs Churn ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(['Monthly Charges', 'Total Charges']):
    df[df['Churn'] == 0][col].plot(kind='hist', bins=40, alpha=0.6,
                                    label='Not Churned', ax=axes[i], color='#2ecc71')
    df[df['Churn'] == 1][col].plot(kind='hist', bins=40, alpha=0.6,
                                    label='Churned', ax=axes[i], color='#e74c3c')
    axes[i].set_title(f'{col} Distribution', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 Churn Rate Across Key Categorical Features ───────────
cat_cols = ['Gender', 'Senior Citizen', 'Partner', 'Dependents',
            'Internet Service', 'Payment Method', 'Paperless Billing']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].mean() * 100
    churn_rate.plot(kind='bar', ax=axes[idx], color='#e74c3c',
                    edgecolor='white', alpha=0.85)
    axes[idx].set_title(f'Churn Rate by {col}', fontsize=11, fontweight='bold')
    axes[idx].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[idx].set_xlabel('')
    axes[idx].tick_params(axis='x', rotation=20)

# Hide unused subplots
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Churn Rate by Key Categorical Features',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.6 Correlation Heatmap (Numeric Features) ───────────────
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0,
            linewidths=0.5, linecolor='white')
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4️⃣ Feature Engineering

In [ ]:
# ── Create New Features ───────────────────────────────────────
df_model = df.copy()

# Average monthly spend
df_model['Avg_Monthly_Spend'] = np.where(
    df_model['Tenure Months'] > 0,
    df_model['Total Charges'] / df_model['Tenure Months'],
    df_model['Monthly Charges']
)

# Tenure buckets
df_model['Tenure_Bucket'] = pd.cut(
    df_model['Tenure Months'],
    bins=[0, 12, 24, 48, 72],
    labels=['New (0-1yr)', 'Growing (1-2yr)',
            'Established (2-4yr)', 'Loyal (4yr+)']
)

# Count of additional services
service_cols = ['Online Security', 'Online Backup', 'Device Protection',
                'Tech Support', 'Streaming TV', 'Streaming Movies']
df_model['Num_Services'] = df_model[service_cols].apply(
    lambda row: sum(row == 'Yes'), axis=1
)

print('✅ New features created:')
print(df_model[['Avg_Monthly_Spend', 'Tenure_Bucket', 'Num_Services']].head())

In [ ]:
# ── Churn Rate by Tenure Bucket ───────────────────────────────
tb_churn = df_model.groupby('Tenure_Bucket', observed=True)['Churn'].mean() * 100

ax = tb_churn.plot(kind='bar', color='#e74c3c', edgecolor='white', figsize=(9, 5))
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Churn Rate by Tenure Bucket', fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure Group')
ax.set_ylabel('Churn Rate (%)')
plt.xticks(rotation=15)

for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Encode Categorical Variables ──────────────────────────────
df_model.drop(columns=['Tenure_Bucket'], inplace=True)  # already captured in Tenure Months

le = LabelEncoder()
cat_cols_model = df_model.select_dtypes(include='object').columns

for col in cat_cols_model:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print(f'✅ Encoded {len(cat_cols_model)} categorical columns')
df_model.head(3)

## 5️⃣ Model Building

In [ ]:
# ── Train / Test Split ────────────────────────────────────────
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size : {X_train.shape[0]:,}')
print(f'Test  size : {X_test.shape[0]:,}')

In [ ]:
# ── Train Both Models ─────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42)
}

results = {}
for name, model in models.items():
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc  if name == 'Logistic Regression' else X_test

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    results[name] = {
        'model':    model,
        'y_pred':   y_pred,
        'y_prob':   y_prob,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc':  roc_auc_score(y_test, y_prob)
    }
    print(f'\n── {name} ──────────────────')
    print(f'  Accuracy : {results[name]["accuracy"]:.4f}')
    print(f'  ROC-AUC  : {results[name]["roc_auc"]:.4f}')
    print(classification_report(y_test, y_pred,
                                target_names=['No Churn', 'Churned']))

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Churn', 'Churned'],
                yticklabels=['No Churn', 'Churned'], ax=ax)
    ax.set_title(f'Confusion Matrix\n{name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curve Comparison ──────────────────────────────────────
plt.figure(figsize=(8, 6))
colors = ['#3498db', '#e74c3c']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{name} (AUC = {res['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Guess')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance (Decision Tree) ───────────────────────
dt_model = results['Decision Tree']['model']
importance_df = pd.DataFrame({
    'Feature':   X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='Importance', y='Feature',
            palette='RdYlGn_r')
plt.title('Top 15 Feature Importances (Decision Tree)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6️⃣ Business Recommendations

Based on our analysis, here are the key findings and actionable recommendations:

---

### 🔴 Key Churn Drivers Found

| Driver | Finding |
|---|---|
| **Contract Type** | Month-to-month customers churn at ~40% vs ~3% for 2-year contracts |
| **Tenure** | Customers in the first 12 months are most at risk |
| **Monthly Charges** | High-charge customers on short contracts churn significantly more |
| **Internet Service** | Fiber optic users churn more than DSL users |
| **No Add-ons** | Customers with no Online Security / Tech Support churn more |

---

### ✅ Recommendations

1. **Offer loyalty discounts** to month-to-month customers after 3 months to push them toward annual contracts.
2. **Onboarding program** — focus retention efforts in the first 12 months (highest churn period).
3. **Bundle add-on services** (Online Security, Tech Support) at discounted rates — customers with more services churn less.
4. **Target high-charge fiber optic users** with retention offers before their 6-month mark.
5. **Deploy the ML model** to score all current customers monthly and proactively contact top 20% churn-risk customers.

In [ ]:
# ── Final Model Summary ───────────────────────────────────────
print('=' * 50)
print('       TELCO CHURN ANALYSIS — SUMMARY')
print('=' * 50)
print(f'Total Customers Analyzed : {len(df):,}')
print(f'Overall Churn Rate       : {df["Churn"].mean()*100:.1f}%')
print()
for name, res in results.items():
    print(f'{name}')
    print(f'  Accuracy  : {res["accuracy"]*100:.2f}%')
    print(f'  ROC-AUC   : {res["roc_auc"]:.4f}')
    print()
print('Top Churn Risk Factors:')
for i, row in importance_df.head(5).iterrows():
    print(f'  {row["Feature"]:30s}  {row["Importance"]:.4f}')
print('=' * 50)